In [1]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
# Folder that contains all your CSV files (sub-folders are searched too)
INPUT_FOLDER = "C:/Users/Lokesh/OneDrive/Documents/dt/datasets"

# Everything the notebook creates goes here
OUTPUT_FOLDER = "C:/Users/Lokesh/OneDrive/Documents/dt/converted"

# Columns to ignore in every file (case-insensitive), e.g. ["id", "index", "record id"]
SKIP_COLUMNS = []

# A column is ignored in a file if more than this share of its values are empty
MAX_EMPTY_FRACTION = 0.90

# Rows whose finished sentence is shorter than this many characters are dropped
MIN_TEXT_LENGTH = 20

# Train / validation / test shares (must add up to 1.0)
SPLIT = (0.8, 0.1, 0.1)
SEED = 42

# If a file name contains the word on the left, its rows get the record type on the right.
# Edit or add to this list to match your file names.
RECORD_TYPE_HINTS = {
    "ntsb": "accident", "crash": "accident", "accident": "accident", "incident": "accident",
    "asrs": "report", "narrative": "report", "report": "report",
    "registry": "aircraft", "aircraft": "aircraft", "plane": "aircraft",
    "airport": "airport", "runway": "airport", "navaid": "airport",
    "airline": "airline", "route": "airline",
    "delay": "operations", "flight": "operations", "ontime": "operations",
    "wildlife": "hazard", "strike": "hazard", "sdr": "hazard", "defect": "hazard",
    "metar": "weather", "weather": "weather",
}

In [3]:
EMPTY_VALUES = {"", "nan", "none", "null", "na", "n/a", "<na>", "nat"}

def read_csv_safely(path):
    """Reads a CSV as text, trying common encodings."""
    last_error = None
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False, dtype=str, on_bad_lines="skip")
        except UnicodeDecodeError as e:
            last_error = e
    raise last_error

def df_to_text(df):
    """Builds one sentence per row from 'Column: value.' pairs, skipping empty values."""
    skip = {c.lower() for c in SKIP_COLUMNS}
    parts = []
    for col in df.columns:
        name = str(col)
        if name.lower() in skip or name.lower().startswith("unnamed"):
            continue
        s = df[col].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
        valid = ~s.str.lower().isin(EMPTY_VALUES)
        s = s.str.replace(r"[.]+$", "", regex=True)
        if valid.mean() < (1 - MAX_EMPTY_FRACTION):
            continue
        label = re.sub(r"[_\s]+", " ", name).strip()
        parts.append((label, s, valid))
    text = pd.Series("", index=df.index, dtype=object)
    for label, s, valid in parts:
        text = text + np.where(valid, label + ": " + s + ". ", "")
    return text.str.strip(), [p[0] for p in parts]

def make_unique_columns(columns):
    """Parquet cannot store two columns with the same name, so repeats get _1, _2 ..."""
    seen, result = {}, []
    for c in columns:
        c = str(c).strip() or "unnamed"
        if c in seen:
            seen[c] += 1
            c = f"{c}_{seen[c]}"
        else:
            seen[c] = 0
        result.append(c)
    return result

def guess_record_type(source_name):
    low = source_name.lower()
    for word, rtype in RECORD_TYPE_HINTS.items():
        if word in low:
            return rtype
    return "other"

In [4]:
in_dir = Path(INPUT_FOLDER)
out_dir = Path(OUTPUT_FOLDER)
(out_dir / "txt").mkdir(parents=True, exist_ok=True)
(out_dir / "parquet").mkdir(parents=True, exist_ok=True)
(out_dir / "parquet_by_file").mkdir(parents=True, exist_ok=True)

csv_files = sorted(in_dir.rglob("*.csv"))
print(f"Found {len(csv_files)} CSV files in {in_dir.resolve()}")

frames, summary = [], []
for path in csv_files:
    source = str(path.relative_to(in_dir).with_suffix("")).replace("\\", "__").replace("/", "__")
    try:
        df = read_csv_safely(path)
    except Exception as e:
        summary.append({"source": source, "rows_in": 0, "rows_out": 0, "columns_used": 0, "note": f"FAILED: {e}"})
        continue
    df.columns = make_unique_columns(df.columns)
    df.to_parquet(out_dir / "parquet_by_file" / f"{source}.parquet", index=False)
    text, used_cols = df_to_text(df)
    keep = text.str.len() >= MIN_TEXT_LENGTH
    kept = text[keep]

    (out_dir / "txt" / f"{source}.txt").write_text("\n".join(kept.tolist()), encoding="utf-8")

    frames.append(pd.DataFrame({
        "text": kept.values,
        "source": source,
        "record_type": guess_record_type(source),
        "event_id": [f"{source}-{i}" for i in kept.index],
    }))
    summary.append({"source": source, "rows_in": len(df), "rows_out": int(keep.sum()),
                    "columns_used": len(used_cols), "note": ""})
    print(f"{source}: {len(df):,} rows -> {int(keep.sum()):,} sentences ({len(used_cols)} columns used)")

pd.DataFrame(summary)

Found 97 CSV files in C:\Users\Lokesh\OneDrive\Documents\dt\datasets
5PctCancels (1) 11: 2 rows -> 2 sentences (1 columns used)
5PctCancels (1): 2 rows -> 2 sentences (1 columns used)
5PctCancels (2) 12: 2 rows -> 2 sentences (1 columns used)
5PctCancels (2): 2 rows -> 2 sentences (1 columns used)
5PctCancels (3) 13: 2 rows -> 2 sentences (1 columns used)
5PctCancels (3): 2 rows -> 2 sentences (1 columns used)
5PctCancels (4) 14: 2 rows -> 2 sentences (1 columns used)
5PctCancels (4): 2 rows -> 2 sentences (1 columns used)
5PctCancels (5) 15: 2 rows -> 2 sentences (1 columns used)
5PctCancels (5): 2 rows -> 2 sentences (1 columns used)
5PctCancels (6) 16: 2 rows -> 2 sentences (1 columns used)
5PctCancels (6): 2 rows -> 2 sentences (1 columns used)
5PctCancels (7): 2 rows -> 2 sentences (1 columns used)
5PctCancels (8): 2 rows -> 2 sentences (1 columns used)
5PctCancels - 2010: 2 rows -> 2 sentences (1 columns used)
5PctCancels - 2015: 2 rows -> 2 sentences (1 columns used)
5PctCancels

,source,rows_in,rows_out,columns_used,note
0,5PctCancels (1) 11,2,2,1,
1,5PctCancels (1),2,2,1,
2,5PctCancels (2) 12,2,2,1,
3,5PctCancels (2),2,2,1,
4,5PctCancels (3) 13,2,2,1,
...,...,...,...,...,...
92,routes,67663,67663,10,
93,US_flights_2023,0,0,0,FAILED: No columns to parse from file
94,Weather Data,8784,8784,8,
95,world_aircraft_accident_summary,607,607,11,


In [5]:
assert abs(sum(SPLIT) - 1.0) < 1e-9, "SPLIT must add up to 1.0"

all_text = pd.concat(frames, ignore_index=True)
before = len(all_text)
all_text = all_text.drop_duplicates(subset="text").reset_index(drop=True)
print(f"Removed {before - len(all_text):,} exact duplicate rows; {len(all_text):,} remain")

# Each source is shuffled and cut on its own, so every source appears in every split
rng = np.random.default_rng(SEED)
def assign_split(group):
    n = len(group)
    order = rng.permutation(n)
    a = int(n * SPLIT[0])
    b = int(n * (SPLIT[0] + SPLIT[1]))
    labels = np.empty(n, dtype=object)
    labels[order[:a]] = "train"
    labels[order[a:b]] = "validation"
    labels[order[b:]] = "test"
    return pd.Series(labels, index=group.index)

all_text["split"] = all_text.groupby("source", group_keys=False).apply(assign_split)

pq = out_dir / "parquet"
all_text.to_parquet(pq / "all_text.parquet", index=False)
for name in ("train", "validation", "test"):
    all_text[all_text["split"] == name].drop(columns="split").to_parquet(pq / f"{name}.parquet", index=False)

print(all_text["split"].value_counts())
all_text.groupby(["record_type", "split"]).size().unstack(fill_value=0)

Removed 109,612 exact duplicate rows; 10,129,645 remain
split
train         8103681
test          1013005
validation    1012959
Name: count, dtype: int64


split,test,train,validation
record_type,,,
accident,47538,380257,47534
aircraft,2525,20196,2525
airline,30580,244611,30576
airport,17122,136951,17119
operations,1173,9372,1171
other,913188,7305267,913156
weather,879,7027,878


In [6]:
train = pd.read_parquet(pq / "train.parquet")
val = pd.read_parquet(pq / "validation.parquet")
test = pd.read_parquet(pq / "test.parquet")

assert len(train) + len(val) + len(test) == len(all_text), "Row counts do not add up"

ids = [set(d["event_id"]) for d in (train, val, test)]
assert not (ids[0] & ids[1]) and not (ids[0] & ids[2]) and not (ids[1] & ids[2]), "An event_id appears in two sets"

expected = {"text", "source", "record_type", "event_id"}
for d in (train, val, test):
    assert set(d.columns) == expected, "Columns differ between files"
    assert (d["text"].str.len() >= MIN_TEXT_LENGTH).all(), "Found a too-short or empty text"

print(f"train {len(train):,} | validation {len(val):,} | test {len(test):,}")
print("All checks passed")

train 8,103,681 | validation 1,012,959 | test 1,013,005
All checks passed


In [7]:
pd.set_option("display.max_colwidth", 250)
for src, grp in all_text.groupby("source"):
    print(f"\n=== {src} ({len(grp):,} rows) ===")
    for t in grp["text"].sample(min(3, len(grp)), random_state=SEED):
        print("-", t[:250])


=== 5PctCancels (2 rows) ===
- January 1990: Regularly Scheduled Flights Canceled 5% or More: SOURCE: Bureau of Transportation Statistics Airline Service Quality Performance Reports.
- January 1990: Regularly Scheduled Flights Canceled 5% or More: Airline: All airlines.

=== 5PctCancels (1) (2 rows) ===
- January 1992: Regularly Scheduled Flights Canceled 5% or More: SOURCE: Bureau of Transportation Statistics Airline Service Quality Performance Reports.
- January 1992: Regularly Scheduled Flights Canceled 5% or More: Airline: All airlines.

=== 5PctCancels (1) 11 (2 rows) ===
- January 2017: Regularly Scheduled Flights Canceled 5% or More: SOURCE: Bureau of Transportation Statistics Airline Service Quality Performance Reports.
- January 2017: Regularly Scheduled Flights Canceled 5% or More: Airline: All airlines.

=== 5PctCancels (2) (2 rows) ===
- January 1993: Regularly Scheduled Flights Canceled 5% or More: SOURCE: Bureau of Transportation Statistics Airline Service Quality Perfor

In [10]:
%whos

Variable              Type           Data/Info
----------------------------------------------
EMPTY_VALUES          set            {'', '<na>', 'n/a', 'none<...>at', 'nan', 'na', 'null'}
INPUT_FOLDER          str            C:/Users/Lokesh/OneDrive/Documents/dt/datasets
MAX_EMPTY_FRACTION    float          0.9
MIN_TEXT_LENGTH       int            20
OUTPUT_FOLDER         str            C:/Users/Lokesh/OneDrive/Documents/dt/converted
Path                  type           <class 'pathlib.Path'>
RECORD_TYPE_HINTS     dict           n=24
SEED                  int            42
SKIP_COLUMNS          list           n=0
SPLIT                 tuple          n=3
all_text              DataFrame      Shape: (10129645, 5)
assign_split          function       <function assign_split at 0x00000262F8CC1F80>
before                int            10239257
csv_files             list           n=97
d                     DataFrame      Shape: (1013005, 4)
df                    DataFrame      Shape: (56, 3)
d

In [11]:
print("TRAINING DATA")
print("Shape:", train.shape)

print("\nCOLUMNS:")
print(train.columns.tolist())

print("\nSAMPLE RECORDS:")
display(train.head(10))

print("\nRECORD TYPE COUNTS:")
print(train["record_type"].value_counts())

TRAINING DATA
Shape: (8103681, 4)

COLUMNS:
['text', 'source', 'record_type', 'event_id']

SAMPLE RECORDS:


,text,source,record_type,event_id
0,January 2017: Regularly Scheduled Flights Canceled 5% or More: Airline: All airlines.,5PctCancels (1) 11,other,5PctCancels (1) 11-0
1,January 1992: Regularly Scheduled Flights Canceled 5% or More: Airline: All airlines.,5PctCancels (1),other,5PctCancels (1)-0
2,January 2020: Regularly Scheduled Flights Canceled 5% or More: SOURCE: Bureau of Transportation Statistics Airline Service Quality Performance Reports.,5PctCancels (2) 12,other,5PctCancels (2) 12-1
3,January 1993: Regularly Scheduled Flights Canceled 5% or More: Airline: All airlines.,5PctCancels (2),other,5PctCancels (2)-0
4,January 1994: Regularly Scheduled Flights Canceled 5% or More: SOURCE: Bureau of Transportation Statistics Airline Service Quality Performance Reports.,5PctCancels (3),other,5PctCancels (3)-1
5,January 2021: Regularly Scheduled Flights Canceled 5% or More: SOURCE: Bureau of Transportation Statistics Airline Service Quality Performance Reports.,5PctCancels (4) 14,other,5PctCancels (4) 14-1
6,January 1996: Regularly Scheduled Flights Canceled 5% or More: Airline: All airlines.,5PctCancels (4),other,5PctCancels (4)-0
7,January 2022: Regularly Scheduled Flights Canceled 5% or More: Airline: All airlines.,5PctCancels (5) 15,other,5PctCancels (5) 15-0
8,January 1997: Regularly Scheduled Flights Canceled 5% or More: Airline: All airlines.,5PctCancels (5),other,5PctCancels (5)-0
9,January 2023: Regularly Scheduled Flights Canceled 5% or More: Airline: All airlines.,5PctCancels (6) 16,other,5PctCancels (6) 16-0



RECORD TYPE COUNTS:
record_type
other         7305267
accident       380257
airline        244611
airport        136951
aircraft        20196
operations       9372
weather          7027
Name: count, dtype: int64


In [12]:
other_data = train[train["record_type"] == "other"]

print("Total Other Records:", len(other_data))

print("\nTop 20 Sources in Other Category:")
display(
    other_data["source"]
    .value_counts()
    .head(20)
    .to_frame("record_count")
)

print("\nSample Records:")
display(other_data[["source", "text"]].sample(20, random_state=42))

Total Other Records: 7305267

Top 20 Sources in Other Category:


,record_count
source,
ALL_ITEMS_DATA_TABLE,7024643
database,139283
Cancelled_Diverted_2023,82834
cleaned_data,24774
aviation- 11,19041
dgca_international_city_pair,12913
passenger_traffic,940
Passengers carried,212
aviation_grievance,206



Sample Records:


,source,text
6127353,ALL_ITEMS_DATA_TABLE,ITEM ID: 279718. ENTITY: Person. ENUMERATOR: 4. ATTRIBUTE: Experience.Flight Crew.Type. VALUE: 4000.
1675544,ALL_ITEMS_DATA_TABLE,ITEM ID: 602570. ENTITY: Person. ENUMERATOR: 1. ATTRIBUTE: Function.Flight Crew. VALUE: Single Pilot.
1581020,ALL_ITEMS_DATA_TABLE,ITEM ID: 344790. ENTITY: Aircraft. ENUMERATOR: 1. ATTRIBUTE: Crew Size.Number Of Crew. VALUE: 2.
627730,ALL_ITEMS_DATA_TABLE,ITEM ID: 249172. ENTITY: Events. ENUMERATOR: 1. ATTRIBUTE: Anomaly.ATC Issue. VALUE: All Types.
5347377,ALL_ITEMS_DATA_TABLE,ITEM ID: 496020. ENTITY: Aircraft. ENUMERATOR: 1. ATTRIBUTE: Flight Phase. VALUE: Descent.
7274699,ALL_ITEMS_DATA_TABLE,ITEM ID: 1241876. ENTITY: Person. ENUMERATOR: 1. ATTRIBUTE: Function.Air Traffic Control. VALUE: Departure. DISPLAY VALUE: Departure.
6466313,ALL_ITEMS_DATA_TABLE,ITEM ID: 862980. ENTITY: Place. ENUMERATOR: 1. ATTRIBUTE: Altitude.MSL.Single Value. VALUE: 34000. DISPLAY VALUE: MSL.
6161055,ALL_ITEMS_DATA_TABLE,ITEM ID: 417625. ENTITY: Events. ENUMERATOR: 1. ATTRIBUTE: Detector.Automation. VALUE: Air Traffic Control.
2366669,ALL_ITEMS_DATA_TABLE,ITEM ID: 299480. ENTITY: Person. ENUMERATOR: 3. ATTRIBUTE: Reference. VALUE: 3.
6282242,ALL_ITEMS_DATA_TABLE,ITEM ID: 89772. ENTITY: Person. ENUMERATOR: 2. ATTRIBUTE: Function.Other.Other. VALUE: Ground Crew.


In [13]:
import pandas as pd
from pathlib import Path

file_path = Path(INPUT_FOLDER) / "ALL_ITEMS_DATA_TABLE.csv"

sample = pd.read_csv(file_path, nrows=20)

print("Columns:")
print(sample.columns.tolist())

print("\nFirst 20 records:")
display(sample)

Columns:
['ITEM_ID', 'ENTITY', 'ENUMERATOR', 'ATTRIBUTE', 'VALUE', 'DISPLAY_VALUE']

First 20 records:


,ITEM_ID,ENTITY,ENUMERATOR,ATTRIBUTE,VALUE,DISPLAY_VALUE
0,209496,Person,1,Experience.Flight Crew.Last 90 Days,180,NaN
1,209496,Person,1,Reference,1,NaN
2,209496,Person,2,Reporter Organization,Air Carrier,NaN
3,209496,Person,4,Reporter Organization,Government,NaN
4,209632,Person,2,Qualification.Flight Crew,Flight Instructor,NaN
5,209632,Person,3,Qualification.Flight Crew,Air Transport Pilot (ATP),NaN
6,209634,Aircraft,1,Crew Size.Number Of Crew,2,NaN
7,209634,Aircraft,1,Flight Phase,Takeoff,NaN
8,209634,Aircraft,1,Mission,Passenger,NaN
9,209498,Place,1,Relative Position.Distance.Nautical Miles,12,NaN


In [14]:
import pandas as pd
from pathlib import Path

file_path = Path(INPUT_FOLDER) / "ALL_ITEMS_DATA_TABLE.csv"
items_df = pd.read_csv(file_path, low_memory=False)

print("Total rows:", len(items_df))
print("Unique ITEM_IDs:", items_df["ITEM_ID"].nunique())

print("\nRows per ITEM_ID:")
print(items_df.groupby("ITEM_ID").size().describe())

print("\nMost common ENTITY types:")
print(items_df["ENTITY"].value_counts())

print("\nDetails for one ITEM_ID:")
sample_id = items_df["ITEM_ID"].iloc[0]

display(
    items_df[items_df["ITEM_ID"] == sample_id]
)

Total rows: 8840247
Unique ITEM_IDs: 184882

Rows per ITEM_ID:
count    184882.000000
mean         47.815618
std          12.595281
min           1.000000
25%          40.000000
50%          47.000000
75%          55.000000
max         224.000000
dtype: float64

Most common ENTITY types:
ENTITY
Person         3539787
Aircraft       2109527
Events         1076183
Place           657887
Environment     429113
Assessments     397596
Time            360738
Component       186159
Report           83257
Name: count, dtype: int64

Details for one ITEM_ID:


,ITEM_ID,ENTITY,ENUMERATOR,ATTRIBUTE,VALUE,DISPLAY_VALUE
0,209496,Person,1,Experience.Flight Crew.Last 90 Days,180,NaN
1,209496,Person,1,Reference,1,NaN
2,209496,Person,2,Reporter Organization,Air Carrier,NaN
3,209496,Person,4,Reporter Organization,Government,NaN
5495,209496,Environment,1,Light,Daylight,NaN
684835,209496,Aircraft,1,Make Model Name,"Medium Large Transport, Low Wing, 2 Turbojet Eng",NaN
1020493,209496,Events,1,Result.General,None Reported / Taken,NaN
1059159,209496,Environment,1,Flight Conditions,VMC,NaN
1059161,209496,Aircraft,1,ATC / Advisory.Tower,ISP,NaN
1059162,209496,Aircraft,1,Crew Size.Number Of Crew,2,NaN


In [15]:
# Select 5 ITEM_IDs for testing
sample_ids = items_df["ITEM_ID"].drop_duplicates().head(5)

sample_data = items_df[
    items_df["ITEM_ID"].isin(sample_ids)
].copy()

# Create readable text for each attribute
sample_data["detail"] = (
    sample_data["ENTITY"].astype(str)
    + " | "
    + sample_data["ATTRIBUTE"].astype(str)
    + " | "
    + sample_data["VALUE"].fillna("").astype(str)
)

# Combine details belonging to the same ITEM_ID
sample_records = (
    sample_data.groupby("ITEM_ID")["detail"]
    .apply(lambda x: " ; ".join(x))
    .reset_index(name="text")
)

print("Combined records:", len(sample_records))
display(sample_records)

Combined records: 5


,ITEM_ID,text
0,209496,Person | Experience.Flight Crew.Last 90 Days | 180 ; Person | Reference | 1 ; Person | Reporter Organization | Air Carrier ; Person | Reporter Organization | Government ; Environment | Light | Daylight ; Aircraft | Make Model Name | Medium Large ...
1,209498,Place | Relative Position.Distance.Nautical Miles | 12 ; Person | Experience.Flight Crew.Last 90 Days | 200 ; Person | Reporter Organization | Air Carrier ; Person | Experience.Flight Crew.Type | 400 ; Person | Function.Flight Crew | Captain ; Pe...
2,209632,Person | Qualification.Flight Crew | Flight Instructor ; Person | Qualification.Flight Crew | Air Transport Pilot (ATP) ; Person | Experience.Flight Crew.Total | 12000 ; Environment | Weather Elements / Visibility.Visibility | 20 ; Aircraft | Fli...
3,209634,Aircraft | Crew Size.Number Of Crew | 2 ; Aircraft | Flight Phase | Takeoff ; Aircraft | Mission | Passenger ; Person | Reporter Organization | Government ; Person | Qualification.Air Traffic Control | Fully Certified ; Aircraft | Make Model Name...
4,209638,Events | Anomaly.Deviation - Procedural | Clearance ; Events | Anomaly.Deviation - Track / Heading | All Types ; Aircraft | Airspace.Class D | VRB ; Person | Reference | 1 ; Environment | Light | Daylight ; Time | Date | 199205 ; Place | Relative...


In [16]:
# Display one complete combined record
first_record = sample_records.iloc[0]

print("ITEM_ID:", first_record["ITEM_ID"])
print("\nCOMPLETE TEXT:\n")
print(first_record["text"])

print("\nCharacter count:", len(first_record["text"]))

ITEM_ID: 209496

COMPLETE TEXT:

Person | Experience.Flight Crew.Last 90 Days | 180 ; Person | Reference | 1 ; Person | Reporter Organization | Air Carrier ; Person | Reporter Organization | Government ; Environment | Light | Daylight ; Aircraft | Make Model Name | Medium Large Transport, Low Wing, 2 Turbojet Eng ; Events | Result.General | None Reported / Taken ; Environment | Flight Conditions | VMC ; Aircraft | ATC / Advisory.Tower | ISP ; Aircraft | Crew Size.Number Of Crew | 2 ; Aircraft | Mission | Passenger ; Person | Function.Flight Crew | First Officer ; Person | Qualification.Flight Crew | Commercial ; Person | Reporter Organization | Government ; Place | Relative Position.Distance.Nautical Miles | 1 ; Person | ASRS Report Number.Accession Number | 209496 ; Person | Experience.Flight Crew.Type | 3000 ; Person | Qualification.Flight Crew | Commercial ; Person | Reference | 2 ; Person | Function.Ground Personnel | Airport Personnel ; Person | Reference | 5 ; Person | Reference 

In [17]:
# Work on a copy
data = items_df.copy()

# Convert values into readable text
def format_detail(row):
    entity = str(row["ENTITY"])
    enumerator = str(row["ENUMERATOR"])
    attribute = str(row["ATTRIBUTE"])

    value = "" if pd.isna(row["VALUE"]) else str(row["VALUE"])
    display = "" if pd.isna(row["DISPLAY_VALUE"]) else str(row["DISPLAY_VALUE"])

    detail = (
        f"{entity} (Enumerator {enumerator}) | "
        f"{attribute} | Value: {value}"
    )

    if display:
        detail += f" | Display Value: {display}"

    return detail

# Create individual text details
data["detail"] = data.apply(format_detail, axis=1)

# Combine details belonging to the same ITEM_ID
combined_records = (
    data.groupby("ITEM_ID", sort=False)["detail"]
    .agg(" ; ".join)
    .reset_index()
)

# Rename columns
combined_records.columns = ["event_id", "text"]

# Add source and category
combined_records["source"] = "ALL_ITEMS_DATA_TABLE"
combined_records["record_type"] = "aviation_report"

print("Total combined records:", len(combined_records))
print("\nColumns:", combined_records.columns.tolist())

display(combined_records.head(3))

Total combined records: 184882

Columns: ['event_id', 'text', 'source', 'record_type']


,event_id,text,source,record_type
0,209496,Person (Enumerator 1) | Experience.Flight Crew.Last 90 Days | Value: 180 ; Person (Enumerator 1) | Reference | Value: 1 ; Person (Enumerator 2) | Reporter Organization | Value: Air Carrier ; Person (Enumerator 4) | Reporter Organization | Value: ...,ALL_ITEMS_DATA_TABLE,aviation_report
1,209632,Person (Enumerator 2) | Qualification.Flight Crew | Value: Flight Instructor ; Person (Enumerator 3) | Qualification.Flight Crew | Value: Air Transport Pilot (ATP) ; Person (Enumerator 1) | Experience.Flight Crew.Total | Value: 12000 ; Environmen...,ALL_ITEMS_DATA_TABLE,aviation_report
2,209634,Aircraft (Enumerator 1) | Crew Size.Number Of Crew | Value: 2 ; Aircraft (Enumerator 1) | Flight Phase | Value: Takeoff ; Aircraft (Enumerator 1) | Mission | Value: Passenger ; Person (Enumerator 3) | Reporter Organization | Value: Government ; P...,ALL_ITEMS_DATA_TABLE,aviation_report


In [18]:
print("TOTAL RECORDS:", len(combined_records))

print("\nMISSING VALUES:")
print(combined_records.isnull().sum())

print("\nDUPLICATE EVENT IDs:")
print(combined_records["event_id"].duplicated().sum())

print("\nEMPTY TEXT RECORDS:")
print((combined_records["text"].str.strip() == "").sum())

print("\nTEXT LENGTH STATISTICS:")
print(combined_records["text"].str.len().describe())

print("\nSAMPLE RECORDS:")
display(combined_records.sample(5, random_state=42))

TOTAL RECORDS: 184882

MISSING VALUES:
event_id       0
text           0
source         0
record_type    0
dtype: int64

DUPLICATE EVENT IDs:
0

EMPTY TEXT RECORDS:
0

TEXT LENGTH STATISTICS:
count    184882.000000
mean       3411.110741
std         941.962828
min          42.000000
25%        2793.000000
50%        3322.000000
75%        3945.000000
max       18236.000000
Name: text, dtype: float64

SAMPLE RECORDS:


,event_id,text,source,record_type
25719,578621,Aircraft (Enumerator 2) | Make Model Name | Value: Commercial Fixed Wing ; Person (Enumerator 1) | Reference | Value: 1 ; Person (Enumerator 5) | Reporter Organization | Value: Government ; Time (Enumerator 1) | Local Time Of Day | Value: 0601-12...,ALL_ITEMS_DATA_TABLE,aviation_report
107291,528010,Person (Enumerator 1) | Experience.Flight Crew.Last 90 Days | Value: 210 ; Person (Enumerator 1) | Function.Flight Crew | Value: Pilot Flying ; Time (Enumerator 1) | Date | Value: 200110 ; Events (Enumerator 1) | Result.General | Value: None Repo...,ALL_ITEMS_DATA_TABLE,aviation_report
16393,349053,Aircraft (Enumerator 1) | Make Model Name | Value: L-1011-1 100/200/250 ; Person (Enumerator 2) | Qualification.Flight Crew | Value: Commercial ; Person (Enumerator 4) | Function.Air Traffic Control | Value: Approach ; Assessments (Enumerator 1) ...,ALL_ITEMS_DATA_TABLE,aviation_report
107169,422845,Place (Enumerator 1) | Locale Reference.Airport | Value: LEX ; Events (Enumerator 1) | Anomaly.Deviation - Procedural | Value: Published Material / Policy ; Place (Enumerator 1) | State Reference | Value: KY ; Events (Enumerator 1) | Anomaly.Devi...,ALL_ITEMS_DATA_TABLE,aviation_report
14194,105090,Environment (Enumerator 1) | Flight Conditions | Value: VMC ; Place (Enumerator 1) | State Reference | Value: WI ; Aircraft (Enumerator 1) | Route In Use.Other | Value: Departure ; Aircraft (Enumerator 2) | Flight Phase | Value: Cruise ; Person (...,ALL_ITEMS_DATA_TABLE,aviation_report


In [19]:
print("MAIN DATASET SHAPE:", all_text.shape)

print("\nMAIN DATASET COLUMNS:")
print(all_text.columns.tolist())

print("\nRECORD TYPE COUNTS:")
print(all_text["record_type"].value_counts())

print("\nSAMPLE EVENT IDs:")
display(all_text[["event_id", "source", "record_type"]].head(10))

MAIN DATASET SHAPE: (10129645, 5)

MAIN DATASET COLUMNS:
['text', 'source', 'record_type', 'event_id', 'split']

RECORD TYPE COUNTS:
record_type
other         9131611
accident       475329
airline        305767
airport        171192
aircraft        25246
operations      11716
weather          8784
Name: count, dtype: int64

SAMPLE EVENT IDs:


,event_id,source,record_type
0,5PctCancels (1) 11-0,5PctCancels (1) 11,other
1,5PctCancels (1) 11-1,5PctCancels (1) 11,other
2,5PctCancels (1)-0,5PctCancels (1),other
3,5PctCancels (1)-1,5PctCancels (1),other
4,5PctCancels (2) 12-0,5PctCancels (2) 12,other
5,5PctCancels (2) 12-1,5PctCancels (2) 12,other
6,5PctCancels (2)-0,5PctCancels (2),other
7,5PctCancels (2)-1,5PctCancels (2),other
8,5PctCancels (3)-0,5PctCancels (3),other
9,5PctCancels (3)-1,5PctCancels (3),other


In [20]:
# Check all source names containing ALL_ITEMS
all_items_sources = all_text[
    all_text["source"].str.contains(
        "ALL_ITEMS_DATA_TABLE",
        case=False,
        na=False
    )
]

print("Existing ALL_ITEMS records:", len(all_items_sources))

print("\nSource names:")
print(all_items_sources["source"].value_counts())

print("\nExisting split distribution:")
print(all_items_sources["split"].value_counts())

print("\nSample existing records:")
display(all_items_sources.head(5))

Existing ALL_ITEMS records: 8780804

Source names:
source
ALL_ITEMS_DATA_TABLE    8780804
Name: count, dtype: int64

Existing split distribution:
split
train         7024643
test           878081
validation     878080
Name: count, dtype: int64

Sample existing records:


,text,source,record_type,event_id,split
570223,ITEM ID: 209496. ENTITY: Person. ENUMERATOR: 1. ATTRIBUTE: Experience.Flight Crew.Last 90 Days. VALUE: 180.,ALL_ITEMS_DATA_TABLE,other,ALL_ITEMS_DATA_TABLE-0,validation
570224,ITEM ID: 209496. ENTITY: Person. ENUMERATOR: 1. ATTRIBUTE: Reference. VALUE: 1.,ALL_ITEMS_DATA_TABLE,other,ALL_ITEMS_DATA_TABLE-1,train
570225,ITEM ID: 209496. ENTITY: Person. ENUMERATOR: 2. ATTRIBUTE: Reporter Organization. VALUE: Air Carrier.,ALL_ITEMS_DATA_TABLE,other,ALL_ITEMS_DATA_TABLE-2,train
570226,ITEM ID: 209496. ENTITY: Person. ENUMERATOR: 4. ATTRIBUTE: Reporter Organization. VALUE: Government.,ALL_ITEMS_DATA_TABLE,other,ALL_ITEMS_DATA_TABLE-3,train
570227,ITEM ID: 209632. ENTITY: Person. ENUMERATOR: 2. ATTRIBUTE: Qualification.Flight Crew. VALUE: Flight Instructor.,ALL_ITEMS_DATA_TABLE,other,ALL_ITEMS_DATA_TABLE-4,train


In [21]:
# Step 1: Remove old individual records
base_text = all_text[
    all_text["source"] != "ALL_ITEMS_DATA_TABLE"
].copy()

# Step 2: Prepare the new combined reports
new_records = combined_records.copy()

# Give the new reports unique IDs
new_records["event_id"] = (
    "ALL_ITEMS_DATA_TABLE_ITEM_"
    + new_records["event_id"].astype(str)
)

# Step 3: Shuffle and assign splits
new_records = new_records.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

total = len(new_records)

train_end = int(total * 0.80)
val_end = int(total * 0.90)

new_records["split"] = "test"
new_records.loc[:train_end - 1, "split"] = "train"
new_records.loc[train_end:val_end - 1, "split"] = "validation"

# Step 4: Arrange columns
new_records = new_records[
    ["text", "source", "record_type", "event_id", "split"]
]

# Step 5: Combine with the remaining aviation data
updated_data = pd.concat(
    [base_text, new_records],
    ignore_index=True
)

print("UPDATED DATASET CREATED SUCCESSFULLY")
print("Total records:", len(updated_data))

print("\nSplit distribution:")
print(updated_data["split"].value_counts())

print("\nRecord types:")
print(updated_data["record_type"].value_counts())

print("\nFinal columns:")
print(updated_data.columns.tolist())

UPDATED DATASET CREATED SUCCESSFULLY
Total records: 1533723

Split distribution:
split
train         1226943
test           153413
validation     153367
Name: count, dtype: int64

Record types:
record_type
accident           475329
other              350807
airline            305767
aviation_report    184882
airport            171192
aircraft            25246
operations          11716
weather              8784
Name: count, dtype: int64

Final columns:
['text', 'source', 'record_type', 'event_id', 'split']


In [22]:
print("TOTAL RECORDS:", len(updated_data))

print("\nMISSING VALUES:")
print(updated_data.isnull().sum())

print("\nDUPLICATE EVENT IDs:")
print(updated_data["event_id"].duplicated().sum())

print("\nEMPTY TEXT RECORDS:")
print((updated_data["text"].str.strip() == "").sum())

print("\nSOURCE DISTRIBUTION:")
print(updated_data["source"].value_counts().head(15))

print("\nSPLIT DISTRIBUTION:")
print(updated_data["split"].value_counts())

print("\nAVIATION REPORT DISTRIBUTION:")
print(
    updated_data[
        updated_data["record_type"] == "aviation_report"
    ]["split"].value_counts()
)

TOTAL RECORDS: 1533723

MISSING VALUES:
text           0
source         0
record_type    0
event_id       0
split          0
dtype: int64

DUPLICATE EVENT IDs:
0

EMPTY TEXT RECORDS:
0

SOURCE DISTRIBUTION:
source
ALL_ITEMS_DATA_TABLE                                184882
database                                            174104
airline_accidents                                   150959
Cancelled_Diverted_2023                             103543
faa_incidents_data                                  100000
Airline Dataset Updated - v2                         98619
Airline Dataset                                      98619
global_airports_aviation_infrastructure_enriched     86242
ntsb_aviation_data                                   85889
airports                                             76367
routes                                               67663
all_plane_crash_data                                 37125
cleaned_data                                         30968
Airline_Scraped_Dat

In [24]:
from pathlib import Path

OUTPUT_FOLDER = Path(
    r"C:\Users\Lokesh\OneDrive\Documents\dt\converted"
)

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

print("OUTPUT FOLDER:", OUTPUT_FOLDER)
print("\nEXISTING FILES:")

for file in OUTPUT_FOLDER.iterdir():
    print(file.name)

OUTPUT FOLDER: C:\Users\Lokesh\OneDrive\Documents\dt\converted

EXISTING FILES:


In [25]:
from pathlib import Path

# Output folder
OUTPUT_FOLDER = Path(
    r"C:\Users\Lokesh\OneDrive\Documents\dt\converted"
)

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# Separate the dataset using the split column
train_final = updated_data[updated_data["split"] == "train"].drop(
    columns=["split"]
)

validation_final = updated_data[
    updated_data["split"] == "validation"
].drop(columns=["split"])

test_final = updated_data[updated_data["split"] == "test"].drop(
    columns=["split"]
)

# Save Parquet files
train_final.to_parquet(
    OUTPUT_FOLDER / "train.parquet", index=False
)

validation_final.to_parquet(
    OUTPUT_FOLDER / "validation.parquet", index=False
)

test_final.to_parquet(
    OUTPUT_FOLDER / "test.parquet", index=False
)

print("PARQUET FILES SAVED SUCCESSFULLY!")
print("Train:", len(train_final))
print("Validation:", len(validation_final))
print("Test:", len(test_final))

PARQUET FILES SAVED SUCCESSFULLY!
Train: 1226943
Validation: 153367
Test: 153413


In [26]:
readme_content = """
---
pretty_name: Aviation Dataset
license: other
task_categories:
- text-generation
language:
- en
---

# Aviation Dataset

## Overview

This dataset contains aviation-related information collected from
multiple datasets. It covers aircraft, airlines, airports, flight
operations, aviation accidents, weather, and aviation reports.

The dataset contains approximately 1.53 million text records.

## Data Source

The dataset used in this project was collected from publicly
available datasets on Kaggle. We downloaded the data and performed
our own data cleaning, preprocessing, and preparation.

## Dataset Splits

| Split | Number of Records |
|---|---:|
| Train | 1,226,943 |
| Validation | 153,367 |
| Test | 153,413 |
| Total | 1,533,723 |

## Columns

- text: Aviation information converted into readable text.
- source: Original dataset name.
- record_type: Category of the aviation record.
- event_id: Unique identifier for each record.

## Record Categories

- Accident
- Airline
- Airport
- Aircraft
- Operations
- Weather
- Aviation reports
- Other aviation information

## File Structure

- train.parquet
- validation.parquet
- test.parquet
- README.md
"""

readme_path = OUTPUT_FOLDER / "README.md"
readme_path.write_text(readme_content.strip(), encoding="utf-8")

print("README CREATED SUCCESSFULLY!")

README CREATED SUCCESSFULLY!


In [27]:
print("FINAL OUTPUT FILES:\n")

for file in sorted(OUTPUT_FOLDER.iterdir()):
    size_mb = file.stat().st_size / (1024 * 1024)
    print(f"{file.name}  |  {size_mb:.2f} MB")

FINAL OUTPUT FILES:

README.md  |  0.00 MB
test.parquet  |  27.01 MB
train.parquet  |  211.27 MB
validation.parquet  |  27.09 MB
